In [1]:
import os
import pandas as pd
from tqdm import tqdm
import concurrent.futures
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne

from typing import Literal
from scipy.signal import butter, filtfilt
from PIL import Image

import torch
from torch import nn
import torch.utils.data
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from pytorch_lightning import LightningDataModule
from pytorch_lightning import LightningModule
import torchmetrics.classification

from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
import torchvision.transforms as transforms
import pytorch_lightning as pl
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
train = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/train.csv')
test = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/test.csv')
sub = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/sample_submission.csv')

In [3]:
temp = pd.read_parquet('/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/3911565283.parquet')

In [4]:
# !rm -r "/kaggle/working/hms-harmful-brain-activity-classification"

In [5]:
dataset_dir = "/kaggle/working/dataset"
eeg_windows_path = os.path.join(dataset_dir, "eeg_windows")
eegs_folder = "kaggle/input/hms-harmful-brain-activity-classification/train_eegs"
spectr_folder = os.path.join(dataset_dir, "spectrograms")
spectr_windows_path = os.path.join(dataset_dir, "spectr_windows")

## Windows extraction and spectrogram image generation

In [6]:
sample_rate = 200
window_len = 50
central_window_start = 20 * sample_rate
central_window_end = 30 * sample_rate


def extract_window(eeg, offset_seconds):
    eeg_sub = eeg.iloc[int(offset_seconds) * sample_rate:(int(offset_seconds) + window_len) * sample_rate]
    labeled_eeg = eeg_sub.iloc[central_window_start:central_window_end]
    
    if labeled_eeg.isnull().values.any():
        return None
    
    return labeled_eeg




spec_zones=['LL','RL','LP','RP']

def generate_spectrogram(spec, specoffset=0.0, id='sample',
                         spec_out=spectr_windows_path,
                         spec_zones=spec_zones,
                         output_filter=0.7):
    
    #take subsample of 600sec (10min)
    spec= spec.loc[(spec.time>=specoffset) & (spec.time<specoffset+600)]
    spec = spec.loc[(spec.time>specoffset+300-5) & (spec.time<=specoffset+300+5)]
    spec = spec.fillna(0)

    #adpat dataset
    spec=spec.set_index('time')
    spec=spec.T
    spec['column']=spec.index.str.split('_', expand=True)

    spec['freq'] = spec.column.apply(lambda x: x[1]).astype(float)
    spec['brainreg'] = spec.column.apply(lambda x: x[0]).astype(str)

    spec=spec.drop('column', axis=1)
    spec.set_index('freq',inplace=True)
    #generate subdatases from brain zones
    subspec=dict()
    for zone in spec_zones:
        subspec[f'{zone}_sub']=spec[spec.brainreg==zone]
        subspec[f'{zone}_sub']= subspec[f'{zone}_sub'].drop('brainreg', axis=1)

    # Genera il grafico dello spettrogramma con dimensioni specificate
    fig, ax = plt.subplots(nrows=len(spec_zones), figsize=(6.4, 6.4), sharex=True)  # 256x256 pixel a 100 dpi
    for row in range(len(spec_zones)):
        data=subspec[f'{spec_zones[row]}_sub']
        ax[row].imshow(data, cmap='turbo',
                       aspect='auto',
                       origin='lower',
                       extent=[data.columns.min(),data.columns.max(),data.index.min(),data.index.max()],
                      vmin=0,vmax=data.max().max()*output_filter)

        ax[row].set_xticks([])
        ax[row].set_yticks([])

    plt.subplots_adjust(hspace=0.01)

    # Salva l'immagine in formato .png
    save_path= f'{spec_out}{id}.png'
    fig.savefig(save_path, bbox_inches='tight', dpi=100)
    plt.close()

    return save_path


## Utils

In [7]:
# utils

# import flatdict
# from omegaconf import OmegaConf


def get_checkpoint(cfg):
    """Returns a ModelCheckpoint callback
    cfg: hydra config
    """
    checkpoint_callback = ModelCheckpoint(monitor='val_loss',
                                          dirpath=cfg['train']['save_path'],
                                          filename=cfg['task'] + '_{epoch}-{step}',
                                          save_last=False)
    return checkpoint_callback

def get_lr_monitor(cfg):
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    return lr_monitor

def get_early_stopping(cfg):
    """Returns an EarlyStopping callback
    cfg: hydra config
    """
    early_stopping_callback = EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=10,
    )
    return early_stopping_callback

class NormalizeEegFeatures:
    def __init__(self, cfg):
        print("Using eeg features normalization")

        # Leggi statistiche da file
#         eegs_stats = pd.read_csv(cfg['dataset']['features_eeg_stats'])

        data = {
            "Min": [0.0],
            "Max": [6239.0371],
            "Mean": [13.051],
            "Std": [102.1672]
        }
        eegs_stats = pd.DataFrame(data)
        train_eegs_min = eegs_stats['Min']
        train_eegs_max = eegs_stats['Max']

        self.train_eegs_min = torch.tensor(train_eegs_min)
        self.train_eegs_max = torch.tensor(train_eegs_max)

    def __call__(self, feature_vec):
        normalized_feature_vec = (feature_vec - self.train_eegs_min) / (self.train_eegs_max - self.train_eegs_min)
        return normalized_feature_vec

class NormalizeSpecFeatures:
    def __init__(self, cfg):
        print("Using spec features normalization")

        # Leggi statistiche da file
        # specs_stats = pd.read_csv(cfg['dataset']['features_spec_stats'])
        data = {
            "Min": [0.0],
            "Max": [559.08],
            "Mean": [16.61],
            "Std": [41.92]
        }
        specs_stats = pd.DataFrame(data)
        train_specs_min = specs_stats['Min']
        train_specs_max = specs_stats['Max']

        self.train_specs_min = torch.tensor(train_specs_min)
        self.train_specs_max = torch.tensor(train_specs_max)

    def __call__(self, feature_vec):
        normalized_feature_vec = (feature_vec - self.train_specs_min) / (self.train_specs_max - self.train_specs_min)
        return normalized_feature_vec

def get_transformations(cfg):
#     stats = pd.read_csv(cfg['dataset']['eeg_stats'])
    import pandas as pd

    data = {
        "Channel": ["Fp1", "F3", "C3", "P3", "F7", "T3", "T5", "O1", "Fz", "Cz", "Pz", "Fp2", "F4", "C4", "P4", "F8", "T4", "T6", "O2", "EKG"],
        "Mean": [0.18, 0.07, 0.21, 0.11, 0.19, 0.13, 0.2, 0.37, -0.1, 0.04, -0.04, 0.23, 0.16, 0.22, 0.19, 0.24, 0.22, 0.19, 0.38, -2.87],
        "Std": [423.16, 318.57, 349.6, 343.86, 346.14, 289.15, 336.5, 316.7, 279.32, 307.89, 346.45, 296.84, 353.18, 342.59, 327.18, 317.36, 311.17, 302.57, 369.44, 13694.39],
        "Min": [-45286.95, -45560.6, -42234.25, -49270.86, -44119.55, -49336.14, -45820.27, -43625.83, -45348.08, -45407.71, -46011.25, -45283.83, -42585.1, -50126.28, -41352.16, -41775.79, -44559.66, -44306.56, -45200.96, -617557.05],
        "Max": [50235.92, 50482.9, 54638.9, 49815.46, 50828.39, 51040.21, 49957.67, 57711.24, 50561.81, 50317.15, 50384.09, 50737.72, 49833.48, 49545.66, 50300.26, 49760.91, 50467.29, 49613.05, 49622.03, 1181368.08]
    }
    stats = pd.DataFrame(data)

    train_mean = stats['Mean']
    train_std = stats['Std']
    train_min = stats['Min']
    train_max = stats['Max']

    train_mean = torch.tensor(train_mean)
    train_std = torch.tensor(train_std)
    train_min = torch.tensor(train_min)
    train_max = torch.tensor(train_max)

    def scale(x):
        if cfg['dataset']['norm_type'] == 'mean_std':
            x = (x - train_mean) / train_std
        elif cfg['dataset']['norm_type'] == 'min_max':
            x = (x - train_min) / (train_max - train_min)
        else:
            raise ValueError("Invalid norm_type")
        return x.T.float()

    eegs_transform = transforms.Compose([
        scale,
    ])

    spectr_transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
    ])

    eeg_features_transform = transforms.Compose([
        NormalizeEegFeatures(cfg)
    ])

    spec_features_transform = transforms.Compose([
        NormalizeSpecFeatures(cfg)
    ])

    return eegs_transform, spectr_transform, eeg_features_transform, spec_features_transform

def log_confusion_matrix_wandb(list_loggers, logger, y_true, preds, class_names):
    # check if wandb is in the list of loggers
    if 'wandb' in list_loggers:
        # logging confusion matrix on wandb
        logger.log({"conf_mat": wandb.plot.confusion_matrix(probs=None, y_true=y_true,
                                                            preds=preds,
                                                            class_names=class_names)})

# def hp_from_cfg(cfg):
#     cfg = OmegaConf.to_container(cfg, resolve=True)
#     return dict(flatdict.FlatDict(cfg, delimiter="/"))

def get_loggers(cfg):
    """Returns a list of loggers
    cfg: hydra config
    """
    loggers = list()
    if cfg['log']['wandb']:
        from pytorch_lightning.loggers import WandbLogger
        import wandb
        hyperparameters = hp_from_cfg(cfg)
        wandb.init(entity=cfg['wandb']['entity'], project=cfg['wandb']['project'])
        wandb.config.update(hyperparameters)
        wandb_logger = WandbLogger()
        loggers.append(wandb_logger)

    return loggers

# Funzione per creare il filtro passabanda
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

# Funzione per applicare il filtro passabanda
def apply_bandpass_filter(data, lowcut=0.5, highcut=40.0, fs=200.0, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data, axis=0)
    return y

## Datamodule

In [8]:
#datamodule
class HMSSignalClassificationDataModule(LightningDataModule):
    def __init__(self, data_dir, mode, freeze, highcut, norm_type, batch_size=32, transform=None):
        super().__init__()

        if mode=="eegsspectr" and freeze:
            print("Using FeatureDataset")
            self.train_dataset = FeatureDataset("train", data_dir, transform)
            self.val_dataset = FeatureDataset("val", data_dir, transform)
            self.test_dataset = FeatureDataset("test", data_dir, transform)   
        else:
            print("Using HMSSignalClassificationDataset")
            self.train_dataset = HMSSignalClassificationDataset("train", data_dir, mode, freeze, highcut, norm_type, transform=transform)
            self.val_dataset = HMSSignalClassificationDataset("val", data_dir, mode, freeze, highcut, norm_type, transform=transform)
            self.test_dataset = HMSSignalClassificationDataset("test", data_dir, mode, freeze, highcut, norm_type, transform=transform)
        self.batch_size = batch_size

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)
    def predict_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)

## Dataset

In [9]:
#dataset

class HMSSignalClassificationDataset(Dataset):
    def __init__(self, stage, data_dir, mode, freeze, highcut, norm_type, transform=None):
        print(f"Loading {stage} dataset in {mode} mode")
        self.stage = stage
        self.data_dir = data_dir
        self.mode = mode
        self.freeze = freeze
        self.highcut = highcut
        self.norm_type = norm_type
        csv_file = os.path.join(data_dir, f"{stage}_{mode}.csv")
        data = pd.read_csv(csv_file)

        self.eeg_ids = data["eeg_id"]
        self.eeg_sub_ids = data["eeg_sub_id"]
        self.eeg_label_offset_seconds = data["eeg_label_offset_seconds"]

        self.label_id = data["label_id"]
        self.expert_consensus = data["expert_consensus"]
        self.seizure_vote = data["seizure_vote"]
        self.lpd_vote = data["lpd_vote"]
        self.gpd_vote = data["gpd_vote"]
        self.lrda_vote = data["lrda_vote"]
        self.grda_vote = data["grda_vote"]
        self.other_vote = data["other_vote"]

        self.class_names = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(self.expert_consensus)

        self.transform = transform
        # passo anche spec_features_transform e eeg_features_transform per la features extraction
        self.eeg_transform, self.spectr_transform, self.eeg_features_transform, self.spec_features_transform = transform 
        self.eeg_transform = None

    def __len__(self):
        return len(self.eeg_ids)

    def __getitem__(self, idx):

        expert_consensus = self.expert_consensus[idx]

        label = self.label_encoder.transform([expert_consensus])[0]  # etichetta
        label = torch.tensor(label, dtype=torch.long)
        label_id = self.label_id[idx]

        if self.mode == 'eegs':
            eeg_file = os.path.join(self.data_dir, f"filtered_eeg_windows_{self.highcut}Hz/{self.stage}/{label_id}.parquet")

            # if self.norm_type == 'mean_std':
            #     eeg_file = os.path.join(self.data_dir, f"standardised_filtered_eeg_windows_{self.highcut}Hz/{self.stage}/{label_id}.csv")

            # eeg_df = pd.read_csv(eeg_file)
            eeg_df = pd.read_parquet(eeg_file)

            # eeg_values = eeg_df.values.astype('float32').T #.
            # eeg_tensor = torch.tensor(eeg_values)

            eeg_tensor = torch.tensor(eeg_df.values)

            if self.eeg_transform:
                eeg = self.eeg_transform(eeg_tensor)

            # transpose eeg tensor
            eeg = eeg_tensor.T

            # print(f"EEG shape: {eeg.shape}")

            return eeg, label

        elif self.mode == 'spectr':
            spectr_file = os.path.join(self.data_dir, "spectr_windows", f"{label_id}.png")
            image = Image.open(spectr_file).convert('RGB')

            if self.spectr_transform:
                image = self.spectr_transform(image)

            return image, label
    
        elif self.mode == 'eegsspectr' and self.freeze==False:
            # eeg_file = os.path.join(self.data_dir, f"{self.stage}_{self.mode}", f"{label_id}.csv")
            eeg_file = os.path.join(self.data_dir, f"filtered_eeg_windows_40Hz/{label_id}.csv")
            eeg_df = pd.read_csv(eeg_file)
            # eeg_values = eeg_df.values.astype('float32').T
            # eeg = torch.tensor(eeg_values)
            if self.eeg_transform:
                eeg = self.eeg_transform(eeg_df)

            spectr_file = os.path.join(self.data_dir, "spectr_windows", f"{label_id}.png")
            image = Image.open(spectr_file).convert('RGB')

            if self.spectr_transform:
                image = self.spectr_transform(image)

            return (eeg, image), label


## Config

In [10]:
# Definisci il contenuto del file YAML come una stringa
cfg = """
# task: 'eegs'          # eegs | spectr | eegsspectr
# task: 'spectr'
task: 'eegsspectr'

dataset:
  data_dir: ./dataset/
  num_classes: 6
  signal_length: 2000
  img_size: 512
  # eeg_stats: "./dataset/eeg_windows_stats.csv"
  eeg_stats: "./dataset/filtered_40Hz_eeg_windows_stats.csv"
  features_eeg_stats: "./dataset/features_eeg_stats.csv"
  features_spec_stats: "./dataset/features_spec_stats.csv"
  highcut: 40
  norm_type: min_max # mean_std | min_max

  # eeg_stats: "./dataset/filtered_40Hz_hq_eeg_windows_stats.csv"
  only_high_quality: False

train:
  save_path: "./checkpoint/"
  seed: -1
  batch_size: 32
  lr: 0.000004
  accelerator: "gpu"
  devices: 1
  max_epochs: 4
  freeze: True         
  feat_comb_mode: 'concat'        # concat | sum | mul | w_sum_eeg | w_sum_spectr 
  # eegs_run_name: "worthy-snowflake-208"
  eegs_run_name: "still-shape-233"
  spectr_run_name: "unique-frost-211"
  eegsspectr_run_name: "vital-frost-279"

checkpoint:
  version: 0

log:
  path: "./logs/"
  wandb: False

wandb:
  entity: martina-marino1996
  project: HMS-kaggle
  tag: ""
"""

# Scrivi il contenuto su un file config.yaml
with open("config.yaml", "w") as file:
    file.write(cfg)

In [11]:
import yaml

# Carica il file YAML
with open("config.yaml", "r") as file:
    cfg = yaml.safe_load(file)

# Preleva un singolo campo (ad esempio, 'task')
task_value = cfg['task']
print(f"Task: {task_value}")

Task: eegsspectr


## Classification

In [12]:
#classification

class HMSEEGClassifierModule(LightningModule):

    def __init__(self, signal_len, num_classes, lr=1e-5, max_epochs=100):
        super().__init__()
        self.save_hyperparameters()
        self.conv1 = nn.Conv1d(in_channels=20, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv1d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        self.fc_input_size = 512 * (
                int(signal_len) // 32)  # 512 out_channels of 5th conv layer and 32 because signal len is reducted
        # after 5 max pooling (2^5 = 32)
        self.fc1 = nn.Linear(self.fc_input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.loss = nn.CrossEntropyLoss()
        self.total_labels = None
        self.total_predictions = None
        self.classes = [i for i in range(num_classes)]

        self.accuracy = torchmetrics.classification.Accuracy(task="multiclass", num_classes=num_classes)
        self.recall = torchmetrics.classification.Recall(task="multiclass", average='weighted', num_classes=num_classes)
        self.precision = torchmetrics.classification.Precision(task="multiclass", average='weighted', num_classes=num_classes)
        self.f1 = torchmetrics.classification.F1Score(task="multiclass", average='weighted', num_classes=num_classes)


    def preprocess(self, x):

        return x

    def extract_features(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        x = self.relu(self.conv4(x))
        x = self.pool(x)
        x = self.relu(self.conv5(x))
        x = self.pool(x)

        x = x.view(-1, self.fc_input_size)

        x = self.relu(self.fc1(x))
        return x

    def forward(self, x):
        # print("Features shape input", x.shape) #[32, 20, 2000]
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        x = self.relu(self.conv4(x))
        x = self.pool(x)
        x = self.relu(self.conv5(x))
        x = self.pool(x)        # after pool -> [32, 512, 62]

        x = x.view(-1, self.fc_input_size)  # before fc1 -> # [32, 31744]

        x = self.relu(self.fc1(x))  # Features shape torch.Size([32, 128])     4.6 M Trainable params
        x = self.fc2(x)

        x = self.softmax(x)

        return x

    def training_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        self._common_step(batch, batch_idx, "val")

    def test_step(self, batch, batch_idx):
        self.eval()
        eegs, labels = batch
        x = self.preprocess(eegs)
        y_hat = self(x)
        predictions = torch.argmax(y_hat, dim=1)
        
        sub_pred = []
        for pred in predictions:
            sub_pred.append(self.classes[pred])

        #log metrics
        self.log('test_accuracy', self.accuracy(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_recall', self.recall(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_precision', self.precision(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_f1', self.f1(predictions, labels), on_step=False, on_epoch=True, logger=True)


    def predict_step(self, batch, batch_idx, dataloader_idx=None):
        eeg, label = batch
        x = self.preprocess(eeg)
        return self(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams.max_epochs, eta_min=1e-5)
        lr_scheduler_config = {
            "scheduler": scheduler,
            "interval": "step",
            "frequency": 1
        }
        return [optimizer], [lr_scheduler_config]

    def _common_step(self, batch, batch_idx, stage):
        signals, labels = batch
        signals = self.preprocess(signals)

        pred = self(signals)
        loss = self.loss(pred, labels)
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True)

        return loss


class HMSSpectrClassifierModule(LightningModule):

    def __init__(self, img_size=512, num_classes=6, lr=1e-5, max_epochs=100):
        super().__init__()
        self.save_hyperparameters()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc_input_size = 512 * (
                img_size // 32) ** 2  # 512 out_channels of 5th conv layer and 32 because signal len is reducted
        # after 5 max pooling (2^5 = 32)
        self.fc1 = nn.Linear(self.fc_input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.loss = nn.CrossEntropyLoss()

        self.accuracy = torchmetrics.classification.Accuracy(task="multiclass", num_classes=num_classes)
        self.recall = torchmetrics.classification.Recall(task="multiclass", average='weighted', num_classes=num_classes)
        self.precision = torchmetrics.classification.Precision(task="multiclass", average='weighted', num_classes=num_classes)
        self.f1 = torchmetrics.classification.F1Score(task="multiclass", average='weighted', num_classes=num_classes)


    def preprocess(self, x):
        return x

    def extract_features(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        x = self.relu(self.conv4(x))
        x = self.pool(x)
        x = self.relu(self.conv5(x))
        x = self.pool(x)

        x = x.view(-1, self.fc_input_size)

        x = self.relu(self.fc1(x))
        return x

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        x = self.relu(self.conv4(x))
        x = self.pool(x)
        x = self.relu(self.conv5(x))
        x = self.pool(x)

        x = x.view(-1, self.fc_input_size)

        x = self.relu(self.fc1(x))
        print("Features shape", x.shape) # Features shape torch.Size([32, 128])     18.3 M Trainable params
        x = self.fc2(x)

        x = self.softmax(x)

        return x

    def training_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        self._common_step(batch, batch_idx, "val")

    def test_step(self, batch, batch_idx):
        self.eval()
        images, labels = batch
        x = self.preprocess(images)
        y_hat = self(x)
        print("y_hat: ", y_hat)
        predictions = torch.argmax(y_hat, dim=1)

        #log metrics
        self.log('test_accuracy', self.accuracy(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_recall', self.recall(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_precision', self.precision(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_f1', self.f1(predictions, labels), on_step=False, on_epoch=True, logger=True)


    def predict_step(self, batch, batch_idx, dataloader_idx=None):
        images, label = batch
        x = self.preprocess(images)
        return self(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams.max_epochs, eta_min=1e-5)
        lr_scheduler_config = {
            "scheduler": scheduler,
            "interval": "step",
            "frequency": 1
        }
        return [optimizer], [lr_scheduler_config]

    def _common_step(self, batch, batch_idx, stage):
        images, labels = batch
        images = self.preprocess(images)

        pred = self(images)
        loss = self.loss(pred, labels)
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True)

        return loss


class HMSEEGSpectrClassifierModule(LightningModule):

    def __init__(self, num_classes, eegs_model_path = "", spectr_model_path = "", freeze=True, lr=1e-5, max_epochs=100, feat_comb_mode=None):
        super().__init__()
        self.save_hyperparameters()

        # load eeg model
        self.eeg_model = HMSEEGClassifierModule.load_from_checkpoint("/kaggle/input/goodplanet/pytorch/default/1/eegs_good-planet-326.ckpt")
        # load spectr model
        self.spectr_model = HMSSpectrClassifierModule.load_from_checkpoint("/kaggle/input/unique-frost/pytorch/default/1/spectr_unique-frost-211.ckpt")

        self.freeze = freeze
        self.feature_comb_mode = feat_comb_mode
        
        if feat_comb_mode == 'concat':
            self.fc1 = nn.Linear(256, 128)
        else:
            self.fc1 = nn.Linear(128, 128)
        self.fc2 = nn.Linear(128, num_classes)

        self.softmax = nn.Softmax(dim=1)
        self.loss = nn.CrossEntropyLoss()

        self.accuracy = torchmetrics.classification.Accuracy(task="multiclass", num_classes=num_classes)
        self.recall = torchmetrics.classification.Recall(task="multiclass", average='weighted', num_classes=num_classes)
        self.precision = torchmetrics.classification.Precision(task="multiclass", average='weighted', num_classes=num_classes)
        self.f1 = torchmetrics.classification.F1Score(task="multiclass", average='weighted', num_classes=num_classes)


    def preprocess(self, x):
        return x

    def forward(self, x):
        eeg_features, spectr_features = x

        if self.freeze:

            # print(f"EEG features shape: {eeg_features.shape}")		#(32, 128)
            # print(f"Spectrogram features shape: {spectr_features.shape}")    #(32,128)
            
            self.eeg_model.freeze()
            self.spectr_model.freeze()
            
            eeg_features = self.eeg_model.extract_features(eeg_features)
            spectr_features = self.spectr_model.extract_features(spectr_features)
            
            eeg_features = (eeg_features - 0.0) / (6239.0371 - 0.0)
            spectr_features = (spectr_features - 0.0) / (559.08 - 0.0)
        

            # switch self.feature_comb_mode
            if self.feature_comb_mode == 'concat':
                combined_features = torch.cat((eeg_features, spectr_features), dim=1)
            elif self.feature_comb_mode == 'sum':
                combined_features = eeg_features + spectr_features
            elif self.feature_comb_mode == 'subtract':
                combined_features = eeg_features - spectr_features
            elif self.feature_comb_mode == 'mul':
                combined_features = eeg_features * spectr_features
            elif self.feature_comb_mode == 'w_sum_eeg':
                combined_features = 0.7 * eeg_features + 0.3 * spectr_features
            elif self.feature_comb_mode == 'w_sum_spectr':
                combined_features = 0.3 * eeg_features + 0.7 * spectr_features
            else:
                raise ValueError("Invalid feature combination mode")

            combined_features = combined_features.float()

            # print(f"Combined features shape: {combined_features.shape}")    #(32,256)
            # # print combined features type
            # print(f"Combined features type: {type(combined_features)}")    #<class 'torch.Tensor'>
            # # print combined features data type
            # print(f"Combined features data type: {combined_features.dtype}") 

            # print("EEG features: ", eeg_features)
            # print("Spectrogram features: ", spectr_features)
            # print("Combined features: ", combined_features)


        else:   
            eeg_features = self.eeg_model.extract_features(eeg_features)
            spectr_features = self.spectr_model.extract_features(spectr_features)

            # print(f"EEG features shape: {eeg_features.shape}")		#(32, 128)
            # print(f"Spectrogram features shape: {spectr_features.shape}")    #(32,128)


            # switch self.feature_comb_mode
            if self.feature_comb_mode == 'concat':
                combined_features = torch.cat((eeg_features, spectr_features), dim=1)
            elif self.feature_comb_mode == 'sum':
                combined_features = eeg_features + spectr_features
            elif self.feature_comb_mode == 'subtract':
                combined_features = eeg_features - spectr_features
            elif self.feature_comb_mode == 'mul':
                combined_features = eeg_features * spectr_features
            elif self.feature_comb_mode == 'weighted_sum':
                combined_features = 0.7 * eeg_features + 0.3 * spectr_features
            else:
                raise ValueError("Invalid feature combination mode")

            combined_features = combined_features.float()

            # print(f"Combined features shape: {combined_features.shape}")    (32,256)
            # # print combined features type
            # print(f"Combined features type: {type(combined_features)}")    #<class 'torch.Tensor'>
            # # print combined features data type
            # print(f"Combined features data type: {combined_features.dtype}")    #torch.float32

        out = self.fc1(combined_features)
        out = self.fc2(out)
        out = self.softmax(out)
    
        return out

    def training_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        self._common_step(batch, batch_idx, "val")

    def test_step(self, batch, batch_idx):
        self.eval()
        (eeg, spectr), labels = batch
        data = (eeg, spectr)
        x = self.preprocess(data)
        y_hat = self(x)
        predictions = torch.argmax(y_hat, dim=1)

        #log metrics
        self.log('test_accuracy', self.accuracy(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_recall', self.recall(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_precision', self.precision(predictions, labels), on_step=False, on_epoch=True, logger=True)
        self.log('test_f1', self.f1(predictions, labels), on_step=False, on_epoch=True, logger=True)


    def predict_step(self, batch, batch_idx, dataloader_idx=None):
        (eeg, spectr), labels = batch
        data = (eeg, spectr)
        x = self.preprocess(data)
        return self(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams.max_epochs,
                                                               eta_min=1e-5)
        lr_scheduler_config = {
            "scheduler": scheduler,
            "interval": "step",
            "frequency": 1
        }
        return [optimizer], [lr_scheduler_config]

    def _common_step(self, batch, batch_idx, stage):
        (eeg, spectr), labels = batch
        data = (eeg, spectr)
        data = self.preprocess(data)

        pred = self(data)
        loss = self.loss(pred, labels)
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True)

        return loss

## Inference

In [13]:
class HMSSignalTestDataModule(LightningDataModule):
    def __init__(self, data_dir, mode, freeze, highcut, norm_type, batch_size=32, transform=None):
        super().__init__()
        print("Using HMSSignalTestDataModule")
        self.test_dataset = HMSSignalTestDataset("test", data_dir, mode, freeze, highcut, norm_type, transform=transform)
        self.batch_size = batch_size

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)
    def predict_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)



class HMSSignalTestDataset(Dataset):
    def __init__(self, stage, data_dir, mode, freeze, highcut, norm_type, transform=None):
        print(f"Loading {stage} dataset in {mode} mode")
        self.stage = stage
        self.data_dir = data_dir
        self.mode = mode
        self.freeze = freeze
        self.highcut = highcut
        self.norm_type = norm_type
        csv_file = os.path.join(data_dir, f"{stage}_{mode}.csv")
        data = pd.read_csv(csv_file)

        self.eeg_ids = data["eeg_id"]
        self.spectr_ids = data["spectrogram_id"]

        self.class_names = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']

        self.transform = transform
        self.eeg_transform, self.spectr_transform, self.eeg_features_transform, self.spec_features_transform = transform 
        #self.eeg_transform = None

    def __len__(self):
        return len(self.eeg_ids)
    

    def __getitem__(self, idx):

        eeg_id = self.eeg_ids[idx]
        spectr_id = self.spectr_ids[idx]

        if self.mode == 'eegs':
            eeg_file = os.path.join(self.data_dir, f"filtered_eeg_windows_{self.highcut}Hz/{eeg_id}.parquet")

            eeg_df = pd.read_parquet(eeg_file)
            eeg_tensor = torch.tensor(eeg_df.values)

            if self.eeg_transform:
                eeg = self.eeg_transform(eeg_tensor)

            eeg = eeg_tensor.T

            return eeg, eeg_id
        
        elif self.mode == 'spectr':
            spectr_file = os.path.join(self.data_dir, "spectr_windows", f"{spectr_id}.png")
            image = Image.open(spectr_file).convert('RGB')

            if self.spectr_transform:
                image = self.spectr_transform(image)

            return image, eeg_id
        
        elif self.mode == 'eegsspectr':
            eeg_file = os.path.join(self.data_dir, f"filtered_eeg_windows_{self.highcut}Hz/{eeg_id}.parquet")

            eeg_df = pd.read_parquet(eeg_file)
            eeg_tensor = torch.tensor(eeg_df.values)

            if self.eeg_transform:
                eeg = self.eeg_transform(eeg_tensor)

            eeg = eeg_tensor.T
            
            spectr_file = os.path.join(self.data_dir, "spectr_windows", f"{spectr_id}.png")
            image = Image.open(spectr_file).convert('RGB')

            if self.spectr_transform:
                image = self.spectr_transform(image)

            return (eeg, image), eeg_id

In [14]:
#test

def main(cfg):
    if cfg['train']['seed'] == -1:
        random_data = os.urandom(4)
        seed = int.from_bytes(random_data, byteorder="big")
        cfg['train']['seed'] = seed
    torch.manual_seed(cfg['train']['seed'])

    callbacks = list()
    callbacks.extend([get_early_stopping(cfg), get_checkpoint(cfg), get_lr_monitor(cfg)])
    loggers = get_loggers(cfg)

    transformations = get_transformations(cfg)
    
    test_df = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/test.csv')


    if cfg['task'] == 'eegs':
        print("Solo EEG")
        ckpt_name = "/kaggle/input/goodplanet/pytorch/default/1/eegs_good-planet-326.ckpt"
        model = HMSEEGClassifierModule.load_from_checkpoint(ckpt_name)

        for id in test_df['eeg_id'].astype(str).values:
            eeg = pd.read_parquet(f"/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/{id}.parquet")
            eeg_window = extract_window(eeg, 0.0)
            filtered_eeg_window = apply_bandpass_filter(eeg_window.values)
            filtered_eeg_window_df = pd.DataFrame(filtered_eeg_window, columns=eeg_window.columns)
            filtered_eeg_window_df = filtered_eeg_window_df.astype(np.float32)
            if not os.path.exists("./submission/filtered_eeg_windows_40Hz"):
                os.makedirs("../submission/filtered_eeg_windows_40Hz")
            filtered_eeg_window_df.to_parquet(f"./submission/filtered_eeg_windows_40Hz/{id}.parquet", index=False)

        test_df.to_csv("/kaggle/working/submission/test_eegs.csv")
        
    elif cfg['task'] == 'spectr':
        print("Solo spectrograms")
        ckpt_name = f"{cfg['save_path']}{cfg['spectr_run_name']}.ckpt"
        model = HMSSpectrClassifierModule.load_from_checkpoint(ckpt_name)
        
        for id in test_df['spectrogram_id'].astype(str).values:
            spectr = pd.read_parquet(f"/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/{id}.parquet")
            if not os.path.exists("./submission/spectr_windows"):
                os.makedirs("../submission/spectr_windows")
            spectr_window = generate_spectrogram(spectr, 0.0, id, "./submission/spectr_windows/", spec_zones, 0.7)

        test_df.to_csv("/kaggle/working/submission/test_eegsspectr.csv")
        
    elif cfg['task'] == 'eegsspectr':
        print("EEG & spectrograms")
        ckpt_name = "/kaggle/input/vital-frost-279/pytorch/default/1/eegsspectr_vital-frost-279.ckpt"
        model = HMSEEGSpectrClassifierModule.load_from_checkpoint(ckpt_name)

        for eeg_id, spectr_id in zip(test_df['eeg_id'].astype(str).values, test_df['spectrogram_id'].astype(str).values):
            eeg = pd.read_parquet(f"/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/{eeg_id}.parquet")
            eeg_window = extract_window(eeg, 0.0)
            filtered_eeg_window = apply_bandpass_filter(eeg_window.values)
            filtered_eeg_window_df = pd.DataFrame(filtered_eeg_window, columns=eeg_window.columns)
            filtered_eeg_window_df = filtered_eeg_window_df.astype(np.float32)
            if not os.path.exists("./submission/filtered_eeg_windows_40Hz"):
                os.makedirs("./submission/filtered_eeg_windows_40Hz")
            filtered_eeg_window_df.to_parquet(f"./submission/filtered_eeg_windows_40Hz/{eeg_id}.parquet", index=False)
            
            
            spectr = pd.read_parquet(f"/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/{spectr_id}.parquet")
            if not os.path.exists("./submission/spectr_windows"):
                os.makedirs("./submission/spectr_windows")
            spectr_window = generate_spectrogram(spectr, 0.0, spectr_id, "./submission/spectr_windows/", spec_zones, 0.7)

        test_df.to_csv("/kaggle/working/submission/test_eegsspectr.csv")
        
        
    data = HMSSignalTestDataModule(
            data_dir="./submission",
            mode=cfg['task'],
            freeze=cfg['train']['freeze'],
            highcut=cfg['dataset']['highcut'],
            norm_type=cfg['dataset']['norm_type'],
            batch_size=cfg['train']['batch_size'],
            transform=transformations
        )
        
    model.eval()

    label_names = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

    # Esegui il modello sul dataset di test
    all_probabilities = []
    eeg_ids = []
    with torch.no_grad():
        for batch in data.test_dataloader():
            
            if cfg['task'] == 'eegs':
                signals, eeg_id_batch = batch
                eeg_id_batch = [int(eeg_id.item()) for eeg_id in eeg_id_batch]
                eeg_ids.extend(eeg_id_batch)

                if isinstance(signals, torch.Tensor):
                    if len(signals.shape) == 2:  # Caso tensore 2D
                        signals = signals.unsqueeze(0)
                elif isinstance(signals, list):
                    signals = torch.stack(signals)

                signals = signals.view(signals.size(0), signals.size(1), -1)
                outputs = model(signals)

            elif cfg['task'] == 'spectr':
                spectr, eeg_id_batch = batch
                eeg_id_batch = [int(eeg_id.item()) for eeg_id in eeg_id_batch]
                eeg_ids.extend(eeg_id_batch)

                if isinstance(signals, torch.Tensor):
                    if len(signals.shape) == 2:  # Caso tensore 2D
                        signals = signals.unsqueeze(0)
                elif isinstance(signals, list):
                    signals = torch.stack(signals)

                    
            elif cfg['task'] == 'eegsspectr':
                (eeg, spectr), eeg_id_batch = batch
                eeg_id_batch = [int(eeg_id.item()) for eeg_id in eeg_id_batch]
                eeg_ids.extend(eeg_id_batch)
#                 print(f"Shape of eeg before modification: {eeg.shape}")
                if isinstance(eeg, torch.Tensor):
                    if len(eeg.shape) == 2:
                        eeg = eeg.unsqueeze(0)
                elif isinstance(eeg, list):
                    eeg = torch.stack(eeg)

#                 print(f"Shape of eeg after modification: {eeg.shape}")

#                 print(f"Shape of spectr before modification: {spectr.shape}")
                if isinstance(spectr, torch.Tensor):
                    if len(spectr.shape) == 2:
                        spectr = spectr.unsqueeze(0)
                elif isinstance(spectr, list):
                    spectr = torch.stack(spectr)

#                 print(f"Shape of spectr after modification: {spectr.shape}")
                outputs = model((eeg, spectr))

            probabilities = F.softmax(outputs, dim=1)  # Calcola le probabilità
            all_probabilities.extend(probabilities.cpu().numpy())

    probabilities_df = pd.DataFrame(all_probabilities, columns=label_names)
    probabilities_df.insert(0, 'eeg_id', eeg_ids)
    probabilities_df.to_csv('submission.csv', index=False)

    print("Probabilità salvate con successo!")


if __name__ == "__main__":
    main(cfg)

Using eeg features normalization
Using spec features normalization
EEG & spectrograms
Using HMSSignalTestDataModule
Loading test dataset in eegsspectr mode
Probabilità salvate con successo!


## Submission

In [15]:
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub)
# SANITY CHECK TO CONFIRM PREDICTIONS SUM TO ONE
sub.iloc[:,-6:].sum(axis=1)

       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  3911565283      0.129563  0.129563  0.129563   0.129563   0.129563   

   other_vote  
0    0.352187  


0    1.0
dtype: float64